[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TigerUnderTheMoon/CF/blob/main/examples/02_graph_diagnostics.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/TigerUnderTheMoon/CF/main/examples/02_graph_diagnostics.ipynb)

Raw notebook URL: [https://raw.githubusercontent.com/TigerUnderTheMoon/CF/main/examples/02_graph_diagnostics.ipynb](https://raw.githubusercontent.com/TigerUnderTheMoon/CF/main/examples/02_graph_diagnostics.ipynb)

# Graph Diagnostics Quick Example

This notebook constructs a NetworkX graph from stored reflection graphs and executes three structural intervention modes (结构干预模式):

- **PRUNE** (single-node removal, 单节点移除): remove one node and its incident edges.
- **CASCADE** (node-plus-descendant removal, 节点及其后继移除): remove a node and all of its descendants.
- **BYPASS** (remove-and-reconnect intervention, 移除并重连下游结构): remove a node while reconnecting its parents directly to its children.

It then visualizes **bottleneck nodes** (瓶颈节点: nodes with both high local utility and high structural necessity).

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/TigerUnderTheMoon/CF.git"
ROOT = Path.cwd()

if not (ROOT / "README.md").exists() or not (ROOT / "src" / "fma").exists():
    clone_dir = ROOT / "CF"
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    ROOT = clone_dir

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Repository root: {ROOT}")

## 1. Load stored reflection graphs

`reflection_graph.json` stores up to 800 reflection DAGs. We load the first 10 for demonstration.

In [ ]:
import pandas as pd
import networkx as nx

from fma.graph.reflection_graph import ReflectionGraph, RemovalMode
from fma.eval.structural_attribution import compute_node_necessity, dataclass_to_dict

graph_path = ROOT / "outputs" / "reflection_graph.json"
graph_payload = json.loads(graph_path.read_text(encoding="utf-8"))
graphs = [ReflectionGraph.from_dict(row) for row in graph_payload["graphs"][:10]]

print(f"Loaded {len(graphs)} reflection graphs")

## 2. Run PRUNE / CASCADE / BYPASS for every node

`compute_node_necessity` performs a graph intervention (图干预) for each node under a chosen removal mode, then compares the graph utility before and after removal. The resulting **structural necessity** (结构必要性) tells us how much the overall graph depends on that specific node.

In [ ]:
rows = []
for graph in graphs:
    for mode in (RemovalMode.PRUNE, RemovalMode.CASCADE, RemovalMode.BYPASS):
        for result in compute_node_necessity(graph, removal_mode=mode):
            rows.append(dataclass_to_dict(result))

necessity = pd.DataFrame(rows)
print(f"Total node-mode rows: {len(necessity)}")
necessity.head()

## 3. Compare modes at the aggregate level

We compute mean structural necessity, the fraction of nodes with positive necessity, and total node counts for each mode.

In [ ]:
mode_summary = (
    necessity.groupby("removal_mode", as_index=False)
    .agg(
        mean_structural_necessity=("necessity_normalized", "mean"),
        positive_fraction=("necessity_normalized", lambda values: float((values > 0).mean())),
        nodes=("node_id", "count"),
    )
)
mode_summary

## 4. Identify bottleneck nodes

A **bottleneck node** (瓶颈节点) has both high local utility (`utility_score`) and high mean structural necessity (`mean_structural_necessity`). We define a simple bottleneck score as the product of (clipped) utility and necessity.

In [ ]:
node_scores = (
    necessity.groupby(["node_id", "trace_id", "step_idx", "taxonomy_label"], as_index=False)
    .agg(
        mean_structural_necessity=("necessity_normalized", "mean"),
        max_structural_necessity=("necessity_normalized", "max"),
        utility_score=("utility_score", "max"),
    )
)

node_scores["bottleneck_score"] = (
    node_scores["utility_score"].clip(lower=0) * node_scores["mean_structural_necessity"].clip(lower=0)
)

node_scores.sort_values("bottleneck_score", ascending=False).head(10)

## 5. Visualize a single reflection graph with bottleneck coloring

We draw the first graph as a NetworkX DiGraph. Node color intensity reflects the bottleneck score computed above.

In [ ]:
import matplotlib.pyplot as plt

selected_graph = graphs[0]
G = nx.DiGraph()
for node_id, node in selected_graph.nodes.items():
    score_row = node_scores[node_scores["node_id"] == node_id]
    bottleneck_score = float(score_row["bottleneck_score"].iloc[0]) if not score_row.empty else 0.0
    G.add_node(node_id, label=node.taxonomy_label, bottleneck_score=bottleneck_score)
for edge in selected_graph.sorted_edges():
    G.add_edge(edge.source, edge.target, edge_type=edge.edge_type)

pos = nx.spring_layout(G, seed=7)
colors = [G.nodes[node]["bottleneck_score"] for node in G.nodes]
labels = {node: G.nodes[node]["label"] for node in G.nodes}

fig, ax = plt.subplots(figsize=(8, 5))
nodes = nx.draw_networkx_nodes(
    G,
    pos,
    node_color=colors,
    cmap="YlOrRd",
    node_size=1200,
    edgecolors="#333333",
    linewidths=1,
    ax=ax,
)
nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle="-|>", width=1.5, ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, ax=ax)
fig.colorbar(nodes, ax=ax, label="Bottleneck score (瓶颈分数)")
ax.set_title("Reflection graph with bottleneck coloring\n（带瓶颈着色的反思图）")
ax.axis("off")
fig.tight_layout()
plt.show()

## Interpretation

The three intervention modes are **complementary diagnostics** (互补诊断):
- PRUNE measures direct local removal sensitivity.
- CASCADE measures downstream dependency when descendants are also removed.
- BYPASS reveals whether downstream flow can remain connected after a node is skipped.

**Bottleneck coloring** is a screening diagnostic (筛选诊断): it highlights candidate structural anchors, but it does **not** prove irreplaceability or hidden causal state.

To explore how failures and costs are audited, continue to [`03_interpreting_audit.ipynb`](03_interpreting_audit.ipynb).